# Flowers GAN experiments with FID / KID / IS (Colab, ~1h GPU)

This notebook trains **only** the **Flowers~102** $64\times 64$ DCGAN from \texttt{GAN\_pytorch.ipynb}, with **longer runs** than the old 10-epoch sweep so optimization actually progresses (~32 steps/epoch at batch 64).

## Metrics

After each run we compute **Fréchet Inception Distance (FID)**, **Kernel Inception Distance (KID)**, and **Inception Score (IS)** with \texttt{torch-fidelity} (\texttt{torch\_fidelity}), comparing **generated images** to a fixed pool of **held-out test** reals (same eval transform, no augmentation). **Lower FID/KID is better**; **higher IS** often indicates sharper/more class-like samples (interpret with care on 64$\times$64 flowers).

**Caveats:** Inception-v3 features are biased toward ImageNet semantics; at 64$\times$64 metrics are noisier than at 299$\times$299. We use a **single seed** and $N$ samples per side---treat numbers as **relative** across rows in your report table, not absolute SOTA.

## Experiments (five runs, ~1 hour total on a mid/high Colab GPU)

| ID | Change |
|----|--------|
| \texttt{FL1\_baseline} | Default: aug on, asymmetric LRs, label smooth, adaptive skip / double-$G$ |
| \texttt{FL2\_no\_aug} | No augmentation |
| \texttt{FL3\_always\_train\_D} | Never skip $D$ (\texttt{d\_skip\_threshold=-1}) |
| \texttt{FL4\_symmetric\_lr} | $\mathrm{lr}_G=\mathrm{lr}_D=$ base lr |
| \texttt{FL5\_no\_label\_smooth} | \texttt{label\_smooth=0} |

## Knobs

- \texttt{FLOWERS\_EPOCHS}: default **150** ($\sim$4.8k steps); reduce on T4 if needed.
- \texttt{N\_GEN}: images per side for FID (default **2048**).
- \texttt{NUM\_WORKERS}: \texttt{0} on Colab if DataLoader hangs.

## Output

After **Run all**, copy the printed \texttt{EXPERIMENTS\_JSON} and optionally \texttt{experiments\_summary.csv}. Fakes live under \texttt{experiment\_outputs/fid\_runs/<id>/fakes/}; reals once under \texttt{experiment\_outputs/fid\_reference/reals/}.


In [ ]:
%pip install -q pytorch-lightning torch-fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 72.6 MB/s eta 0:00:00


In [ ]:
import json
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from pytorch_lightning.callbacks import Callback
from torch_fidelity import calculate_metrics
from torch.utils.data import ConcatDataset, DataLoader
from torchvision.datasets import Flowers102

random_seed = 42
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
torch.set_float32_matmul_precision("medium")

# --- budget: ~1h total for 5 runs + FID on a typical Colab GPU ---
FLOWERS_EPOCHS = 150
BATCH_FLOWERS = 64
N_GEN = 2048  # same count for reals (from test) and fakes
NUM_WORKERS = 2  # use 0 if Colab workers hang

DATA_DIR = "./data"
OUT_DIR = Path("experiment_outputs")
GRID_DIR = OUT_DIR / "experiment_grids"
FID_RUNS = OUT_DIR / "fid_runs"
REALS_DIR = OUT_DIR / "fid_reference" / "reals"
OUT_DIR.mkdir(exist_ok=True)
GRID_DIR.mkdir(exist_ok=True)
FID_RUNS.mkdir(exist_ok=True)
REALS_DIR.parent.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", GPU_NAME)


Device: cuda | NVIDIA A100-SXM4-40GB


In [ ]:
class FlowersDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_dir=DATA_DIR,
        batch_size=BATCH_FLOWERS,
        num_workers=NUM_WORKERS,
        image_size=64,
        combine_train_val=True,
        use_augmentation=True,
    ):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.image_size = image_size
        self.combine_train_val = combine_train_val
        self.use_augmentation = use_augmentation

        self.eval_transform = transforms.Compose(
            [
                transforms.Resize(image_size),
                transforms.CenterCrop(image_size),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )
        base_transform = list(self.eval_transform.transforms)
        if use_augmentation:
            self.train_transform = transforms.Compose(
                [
                    transforms.Resize(int(image_size * 1.1)),
                    transforms.RandomCrop(image_size),
                    transforms.RandomHorizontalFlip(p=0.5),
                    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                    transforms.ToTensor(),
                    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                ]
            )
        else:
            self.train_transform = transforms.Compose(base_transform)

    def prepare_data(self):
        Flowers102(self.data_dir, split="train", download=True)
        Flowers102(self.data_dir, split="val", download=True)
        Flowers102(self.data_dir, split="test", download=True)

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            flowers_train = Flowers102(self.data_dir, split="train", transform=self.train_transform)
            if self.combine_train_val:
                flowers_val = Flowers102(self.data_dir, split="val", transform=self.train_transform)
                self.flowers_train = ConcatDataset([flowers_train, flowers_val])
            else:
                self.flowers_train = flowers_train
        if stage == "test" or stage is None:
            self.flowers_test = Flowers102(self.data_dir, split="test", transform=self.eval_transform)

    def train_dataloader(self):
        return DataLoader(
            self.flowers_train,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=bool(self.num_workers),
        )

    def test_dataloader(self):
        return DataLoader(
            self.flowers_test,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            persistent_workers=bool(self.num_workers),
        )


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels=3, image_size=64):
        super().__init__()
        self.channels = channels
        self.image_size = image_size
        self.conv1 = nn.Conv2d(channels, 64, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        self.fc = nn.Linear(4 * 4 * 512, 1)

    def forward(self, x):
        x = F.leaky_relu(self.conv1(x), 0.2)
        x = F.leaky_relu(self.bn2(self.conv2(x)), 0.2)
        x = F.leaky_relu(self.bn3(self.conv3(x)), 0.2)
        x = F.leaky_relu(self.bn4(self.conv4(x)), 0.2)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return torch.sigmoid(x)


class Generator(nn.Module):
    def __init__(self, latent_dim, channels=3, image_size=64):
        super().__init__()
        self.image_size = image_size
        self.lin1 = nn.Linear(latent_dim, 4 * 4 * 512)
        self.ct1 = nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(256)
        self.ct2 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.ct3 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.ct4 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(32)
        self.conv = nn.Conv2d(32, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.lin1(x)
        x = F.relu(x)
        x = x.view(-1, 512, 4, 4)
        x = F.relu(self.bn1(self.ct1(x)))
        x = F.relu(self.bn2(self.ct2(x)))
        x = F.relu(self.bn3(self.ct3(x)))
        x = F.relu(self.bn4(self.ct4(x)))
        x = torch.tanh(self.conv(x))
        return x


class GANExperiment(pl.LightningModule):
    # Same logic as GAN_pytorch / models.py; Flowers 64x64 only in this notebook.

    def __init__(
        self,
        latent_dim=100,
        channels=3,
        image_size=64,
        lr=0.0002,
        label_smooth=0.1,
        lr_g_mult=2.0,
        lr_d_mult=0.5,
        beta1=0.5,
        d_skip_threshold=0.6,
        d_double_g_threshold=0.4,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.automatic_optimization = False
        self.generator = Generator(latent_dim=latent_dim, channels=channels, image_size=image_size)
        self.discriminator = Discriminator(channels=channels, image_size=image_size)
        self.validation_z = torch.randn(8, latent_dim)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, z):
        return self.generator(z)

    def adversarial_loss(self, y_hat, y):
        return F.binary_cross_entropy(y_hat, y)

    def training_step(self, batch, batch_idx):
        opt_g, opt_d = self.optimizers()
        real_imgs, _ = batch
        batch_size = real_imgs.size(0)
        z = torch.randn(batch_size, self.hparams.latent_dim, device=real_imgs.device)

        opt_d.zero_grad()
        y_hat_real = self.discriminator(real_imgs)
        y_real = torch.ones(batch_size, 1, device=real_imgs.device) * (1.0 - self.hparams.label_smooth)
        real_loss = self.adversarial_loss(y_hat_real, y_real)
        fake_imgs_det = self(z).detach()
        y_hat_fake = self.discriminator(fake_imgs_det)
        y_fake = torch.zeros(batch_size, 1, device=real_imgs.device) + self.hparams.label_smooth
        fake_loss = self.adversarial_loss(y_hat_fake, y_fake)
        d_loss = (real_loss + fake_loss) / 2

        thr_skip = self.hparams.d_skip_threshold
        if thr_skip is None or thr_skip < 0 or d_loss.item() >= thr_skip:
            self.manual_backward(d_loss)
            opt_d.step()

        with torch.no_grad():
            pred_real = (y_hat_real > 0.5).float()
            pred_fake = (y_hat_fake < 0.5).float()
            d_acc = (pred_real.mean() + pred_fake.mean()) / 2

        opt_g.zero_grad()
        fake_imgs = self(z)
        y_hat = self.discriminator(fake_imgs)
        y = torch.ones(batch_size, 1, device=real_imgs.device)
        g_loss = self.adversarial_loss(y_hat, y)
        self.manual_backward(g_loss)
        opt_g.step()

        thr_dg = self.hparams.d_double_g_threshold
        if thr_dg is not None and thr_dg >= 0 and d_loss.item() < thr_dg:
            opt_g.zero_grad()
            z2 = torch.randn(batch_size, self.hparams.latent_dim, device=real_imgs.device)
            fake2 = self(z2)
            y_hat2 = self.discriminator(fake2)
            g_loss2 = self.adversarial_loss(y_hat2, y)
            self.manual_backward(g_loss2)
            opt_g.step()
            g_loss = (g_loss + g_loss2) / 2

        self.log("g_loss", g_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("d_loss", d_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("d_acc", d_acc, on_step=False, on_epoch=True, prog_bar=True)
        return {"loss": g_loss + d_loss}

    def configure_optimizers(self):
        lr_g = self.hparams.lr * self.hparams.lr_g_mult
        lr_d = self.hparams.lr * self.hparams.lr_d_mult
        b1 = self.hparams.beta1
        opt_g = torch.optim.Adam(self.generator.parameters(), lr=lr_g, betas=(b1, 0.999))
        opt_d = torch.optim.Adam(self.discriminator.parameters(), lr=lr_d, betas=(b1, 0.999))
        return [opt_g, opt_d], []


class LossHistoryCallback(Callback):
    def __init__(self):
        self.g_loss_epochs = []
        self.d_loss_epochs = []
        self.d_acc_epochs = []

    def on_train_epoch_end(self, trainer, pl_module):
        g = trainer.callback_metrics.get("g_loss")
        d = trainer.callback_metrics.get("d_loss")
        a = trainer.callback_metrics.get("d_acc")
        if g is not None:
            self.g_loss_epochs.append(float(g.detach().cpu() if hasattr(g, "detach") else g))
        if d is not None:
            self.d_loss_epochs.append(float(d.detach().cpu() if hasattr(d, "detach") else d))
        if a is not None:
            self.d_acc_epochs.append(float(a.detach().cpu() if hasattr(a, "detach") else a))


def save_sample_grid(model, path: Path, n_show: int = 8):
    model.eval()
    dev = next(model.parameters()).device
    z = model.validation_z.to(dev)
    with torch.no_grad():
        samples = model(z).cpu()
    samples = (samples + 1) / 2.0
    samples = torch.clamp(samples, 0, 1)
    c = model.hparams.channels
    fig, axes = plt.subplots(2, 4, figsize=(8, 4))
    for i, ax in enumerate(axes.flat):
        if i >= n_show:
            break
        if c == 1:
            ax.imshow(samples[i, 0].numpy(), cmap="gray", interpolation="none")
        else:
            ax.imshow(samples[i].permute(1, 2, 0).numpy(), interpolation="none")
        ax.axis("off")
    plt.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    model.train()


In [ ]:
def export_real_pool(dataset, out_dir: Path, n: int) -> int:
    # Save first n test images as PNG in [0,1] (deterministic).
    out_dir.mkdir(parents=True, exist_ok=True)
    n = min(n, len(dataset))
    for f in out_dir.glob("*.png"):
        f.unlink()
    for i in range(n):
        img, _ = dataset[i]
        path = out_dir / f"{i:05d}.png"
        torchvision.utils.save_image((img + 1) / 2, path)
    return n


def generate_fake_folder(model, out_dir: Path, n: int, batch_size: int = 64) -> None:
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model.eval()
    dev = next(model.parameters()).device
    saved = 0
    while saved < n:
        bs = min(batch_size, n - saved)
        z = torch.randn(bs, model.hparams.latent_dim, device=dev)
        with torch.no_grad():
            fake = model.generator(z)
        fake = torch.clamp((fake + 1) / 2, 0, 1)
        for j in range(bs):
            torchvision.utils.save_image(fake[j], out_dir / f"{saved + j:05d}.png")
        saved += bs
    model.train()


def compute_fid_kid_isc(fakes_dir: Path, reals_dir: Path) -> dict:
    m = calculate_metrics(
        input1=str(fakes_dir),
        input2=str(reals_dir),
        cuda=torch.cuda.is_available(),
        isc=True,
        fid=True,
        kid=True,
        verbose=False,
    )
    return {
        "fid": m.get("frechet_inception_distance"),
        "kid_mean": m.get("kernel_inception_distance_mean"),
        "kid_std": m.get("kernel_inception_distance_std"),
        "isc_mean": m.get("inception_score_mean"),
        "isc_std": m.get("inception_score_std"),
    }


def ensure_real_reference_pool(dm: FlowersDataModule, n: int) -> int:
    # Export test-set reals once (same folder reused for all runs).
    dm.prepare_data()
    dm.setup("test")
    have = len(list(REALS_DIR.glob("*.png"))) if REALS_DIR.exists() else 0
    if have >= n:
        return min(n, have)
    return export_real_pool(dm.flowers_test, REALS_DIR, n)


def run_flowers_experiment(
    experiment_id: str,
    max_epochs: int,
    model_kwargs: dict,
    flowers_use_augmentation: bool,
    n_real_fake: int,
):
    pl.seed_everything(random_seed, workers=True)

    dm = FlowersDataModule(
        batch_size=BATCH_FLOWERS,
        num_workers=NUM_WORKERS,
        use_augmentation=flowers_use_augmentation,
    )
    dm.setup("fit")
    steps_per_epoch = len(dm.train_dataloader())

    model = GANExperiment(channels=3, image_size=64, **model_kwargs)
    hist = LossHistoryCallback()

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        enable_checkpointing=False,
        logger=False,
        enable_progress_bar=True,
        log_every_n_steps=5,
        callbacks=[hist],
    )

    t_train = time.perf_counter()
    trainer.fit(model, dm)
    train_seconds = time.perf_counter() - t_train

    g_hist = hist.g_loss_epochs
    d_hist = hist.d_loss_epochs
    dacc_hist = hist.d_acc_epochs
    final_g = g_hist[-1] if g_hist else float("nan")
    final_d = d_hist[-1] if d_hist else float("nan")
    final_d_acc = dacc_hist[-1] if dacc_hist else float("nan")
    k = min(3, len(g_hist))
    mean_g3 = sum(g_hist[-k:]) / k if k else float("nan")
    mean_d3 = sum(d_hist[-k:]) / k if k else float("nan")
    mean_d_acc3 = sum(dacc_hist[-k:]) / k if k and dacc_hist else float("nan")

    grid_path = GRID_DIR / f"{experiment_id}.png"
    save_sample_grid(model, grid_path)

    fake_dir = FID_RUNS / experiment_id / "fakes"
    t_met = time.perf_counter()
    generate_fake_folder(model, fake_dir, n=n_real_fake, batch_size=BATCH_FLOWERS)
    metrics = compute_fid_kid_isc(fake_dir, REALS_DIR)
    metrics_seconds = time.perf_counter() - t_met

    record = {
        "experiment_id": experiment_id,
        "dataset": "flowers",
        "max_epochs": max_epochs,
        "batch_size": BATCH_FLOWERS,
        "steps_per_epoch": steps_per_epoch,
        "n_gen": n_real_fake,
        "n_real_reference": n_real_fake,
        "train_seconds": round(train_seconds, 2),
        "metrics_seconds": round(metrics_seconds, 2),
        "seconds": round(train_seconds + metrics_seconds, 2),
        "hardware_note": GPU_NAME,
        "final_g_loss": round(final_g, 6),
        "final_d_loss": round(final_d, 6),
        "final_d_acc": round(final_d_acc, 6),
        "mean_g_loss_last_3_epochs": round(mean_g3, 6),
        "mean_d_loss_last_3_epochs": round(mean_d3, 6),
        "mean_d_acc_last_3_epochs": round(mean_d_acc3, 6),
        "flowers_use_augmentation": flowers_use_augmentation,
        **{f"hparam_{k}": v for k, v in model_kwargs.items()},
        "fid": None if metrics["fid"] is None else round(float(metrics["fid"]), 4),
        "kid_mean": None if metrics["kid_mean"] is None else round(float(metrics["kid_mean"]), 6),
        "kid_std": None if metrics["kid_std"] is None else round(float(metrics["kid_std"]), 6),
        "isc_mean": None if metrics["isc_mean"] is None else round(float(metrics["isc_mean"]), 4),
        "isc_std": None if metrics["isc_std"] is None else round(float(metrics["isc_std"]), 4),
        "g_loss_history": [round(x, 6) for x in g_hist],
        "d_loss_history": [round(x, 6) for x in d_hist],
        "d_acc_history": [round(x, 6) for x in dacc_hist],
        "sample_grid_png": str(grid_path.as_posix()),
        "fakes_dir": str(fake_dir.as_posix()),
        "reals_dir": str(REALS_DIR.as_posix()),
    }
    return record


BASE = dict(
    lr=0.0002,
    label_smooth=0.1,
    latent_dim=100,
    lr_g_mult=2.0,
    lr_d_mult=0.5,
    beta1=0.5,
    d_skip_threshold=0.6,
    d_double_g_threshold=0.4,
)

# (experiment_id, epochs, model_kw overrides, use_augmentation)
EXPERIMENTS = [
    ("FL1_baseline", FLOWERS_EPOCHS, {**BASE}, True),
    ("FL2_no_aug", FLOWERS_EPOCHS, {**BASE}, False),
    ("FL3_always_train_D", FLOWERS_EPOCHS, {**BASE, "d_skip_threshold": -1.0}, True),
    ("FL4_symmetric_lr", FLOWERS_EPOCHS, {**BASE, "lr_g_mult": 1.0, "lr_d_mult": 1.0}, True),
    ("FL5_no_label_smooth", FLOWERS_EPOCHS, {**BASE, "label_smooth": 0.0}, True),
]


In [ ]:
# Export held-out test reals once (deterministic first N_GEN images)
_dm_ref = FlowersDataModule(batch_size=BATCH_FLOWERS, num_workers=NUM_WORKERS, use_augmentation=True)
n_exported = ensure_real_reference_pool(_dm_ref, N_GEN)
print(f"FID reference reals: {n_exported} images -> {REALS_DIR}")

results = []
for eid, ep, kw, f_aug in EXPERIMENTS:
    rec = run_flowers_experiment(eid, ep, kw, f_aug, n_real_fake=N_GEN)
    results.append(rec)
    print(
        f"Done {eid} | train {rec['train_seconds']}s + metrics {rec['metrics_seconds']}s | "
        f"FID {rec['fid']} | KID {rec['kid_mean']} | IS {rec['isc_mean']}"
    )

import pandas as pd

slim = []
for r in results:
    row = {k: v for k, v in r.items() if k not in ("g_loss_history", "d_loss_history", "d_acc_history")}
    slim.append(row)
df = pd.DataFrame(slim)
csv_path = OUT_DIR / "experiments_summary.csv"
df.to_csv(csv_path, index=False)
print("\nSaved:", csv_path)
try:
    display(df)
except NameError:
    print(df.to_string())

print("\n" + "=" * 60)
print("COPY FOR REPORT: EXPERIMENTS_JSON")
print("=" * 60)
print(json.dumps(results, indent=2))
print("=" * 60)

total_train = sum(r["train_seconds"] for r in results)
total_met = sum(r["metrics_seconds"] for r in results)
print(f"\nTotal training (sum): {total_train/60:.1f} min | Total FID pipeline (sum): {total_met/60:.1f} min")
print(f"Total wall (sum of per-run train+metrics): {(total_train + total_met)/60:.1f} min")


Epoch 149/149 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32/32 0:00:04 • 0:00:00 7.07it/s g_loss: 0.770 d_loss: 0.706 d_acc:
                                                                                 0.489                             

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=150` reached.


Done FL5_no_label_smooth | train 710.29s + metrics 16.05s | FID 239.8794 | KID 0.213806 | IS 2.3445

Saved: experiment_outputs/experiments_summary.csv


,experiment_id,dataset,max_epochs,batch_size,steps_per_epoch,n_gen,n_real_reference,train_seconds,metrics_seconds,seconds,...,hparam_d_skip_threshold,hparam_d_double_g_threshold,fid,kid_mean,kid_std,isc_mean,isc_std,sample_grid_png,fakes_dir,reals_dir
0,FL1_baseline,flowers,150,64,32,2048,2048,673.65,17.40,691.05,...,0.6,0.4,279.1048,0.272188,0.002090,2.2310,0.0974,experiment_outputs/experiment_grids/FL1_baseli...,experiment_outputs/fid_runs/FL1_baseline/fakes,experiment_outputs/fid_reference/reals
1,FL2_no_aug,flowers,150,64,32,2048,2048,590.65,15.87,606.52,...,0.6,0.4,307.7742,0.330806,0.002856,1.7486,0.0428,experiment_outputs/experiment_grids/FL2_no_aug...,experiment_outputs/fid_runs/FL2_no_aug/fakes,experiment_outputs/fid_reference/reals
2,FL3_always_train_D,flowers,150,64,32,2048,2048,691.06,14.98,706.04,...,-1.0,0.4,302.1805,0.301162,0.002499,2.0830,0.1104,experiment_outputs/experiment_grids/FL3_always...,experiment_outputs/fid_runs/FL3_always_train_D...,experiment_outputs/fid_reference/reals
3,FL4_symmetric_lr,flowers,150,64,32,2048,2048,688.97,15.09,704.05,...,0.6,0.4,210.9616,0.180458,0.002075,3.4286,0.1134,experiment_outputs/experiment_grids/FL4_symmet...,experiment_outputs/fid_runs/FL4_symmetric_lr/f...,experiment_outputs/fid_reference/reals
4,FL5_no_label_smooth,flowers,150,64,32,2048,2048,710.29,16.05,726.34,...,0.6,0.4,239.8794,0.213806,0.002027,2.3445,0.0539,experiment_outputs/experiment_grids/FL5_no_lab...,experiment_outputs/fid_runs/FL5_no_label_smoot...,experiment_outputs/fid_reference/reals



COPY FOR REPORT: EXPERIMENTS_JSON
[
  {
    "experiment_id": "FL1_baseline",
    "dataset": "flowers",
    "max_epochs": 150,
    "batch_size": 64,
    "steps_per_epoch": 32,
    "n_gen": 2048,
    "n_real_reference": 2048,
    "train_seconds": 673.65,
    "metrics_seconds": 17.4,
    "seconds": 691.05,
    "hardware_note": "NVIDIA A100-SXM4-40GB",
    "final_g_loss": 0.777539,
    "final_d_loss": 0.704019,
    "final_d_acc": 0.505392,
    "mean_g_loss_last_3_epochs": 0.764655,
    "mean_d_loss_last_3_epochs": 0.70014,
    "mean_d_acc_last_3_epochs": 0.511275,
    "flowers_use_augmentation": true,
    "hparam_lr": 0.0002,
    "hparam_label_smooth": 0.1,
    "hparam_latent_dim": 100,
    "hparam_lr_g_mult": 2.0,
    "hparam_lr_d_mult": 0.5,
    "hparam_beta1": 0.5,
    "hparam_d_skip_threshold": 0.6,
    "hparam_d_double_g_threshold": 0.4,
    "fid": 279.1048,
    "kid_mean": 0.272188,
    "kid_std": 0.00209,
    "isc_mean": 2.231,
    "isc_std": 0.0974,
    "g_loss_history": [
      3

## Optional: plot $L_G$ and $d_{\mathrm{acc}}$ for all runs

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(10, 4))
# for r in results:
#     ax[0].plot(r["g_loss_history"], label=r["experiment_id"])
#     ax[1].plot(r["d_acc_history"], label=r["experiment_id"])
# ax[0].set_title("G loss"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=7)
# ax[1].set_title("D accuracy (batch heuristic)"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=7)
# plt.tight_layout()
# plt.savefig(OUT_DIR / "flowers_loss_dacc_compare.png", dpi=150)
# plt.show()
print("(Optional plotting cell — uncomment to use)")
